# Sentinel-Sec Pipeline Workshop

Welcome! This hands-on notebook walks you through the core computer-vision pipeline that powers the Sentinel-Sec webapp.

You will:
- Set up the environment and verify GPU/runtime availability
- Detect faces with InsightFace (FaceAnalysis)
- Compute embeddings and match with cosine similarity
- Analyze demographics with DeepFace (optional)
- Persist profiles, encodings, and detections to SQLite
- Batch process images and sample video frames with auto-enrollment
- Visualize analytics and prepare artifacts for the Streamlit app

## 1. Environment Setup and Dependency Checks

This section verifies that required libraries are available and reports versions. If GPU is present, we'll prefer it for InsightFace.

If a package is missing, install it in your environment (you can restart the kernel afterwards).

In [ ]:
# Versions and runtime checks
import importlib, sys
import platform
print(f"Python: {platform.python_version()}")

packages = [
    "opencv-python", "numpy", "pandas", "matplotlib", "plotly", 
    "insightface", "deepface", "onnxruntime", "onnxruntime-gpu", "torch"
]

installed = {}
for pkg in packages:
    try:
        modname = pkg.replace('-', '_')
        mod = importlib.import_module(modname)
        ver = getattr(mod, '__version__', 'unknown')
        installed[pkg] = ver
    except Exception as e:
        installed[pkg] = None

for k,v in installed.items():
    print(f"{k:18s} -> {v if v else 'NOT INSTALLED'}")

# GPU availability checks
try:
    import torch
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
except Exception as e:
    print("torch not available")

try:
    import onnxruntime as ort
    print("ONNX providers:", ort.get_available_providers())
except Exception as e:
    print("onnxruntime not available")

## 2. Project Path Setup and Imports

We add the project root to `sys.path` so imports like `lib.face_detector` work from this notebook.

In [ ]:
# Add repo root to sys.path and import project modules
import os, sys
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd()))
if os.path.basename(ROOT).lower() != 'sentinel-sec':
    # if running from docs/workshop, go two levels up
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print("Project root:", ROOT)
sys.path.append(ROOT)

from lib.database import DatabaseManager
from lib.face_detector import FaceDetector, create_test_image
from lib.face_analyzer import FaceAnalyzer

## 3. Configure Model Cache Directories and Device Selection

We mirror the app's environment variables so InsightFace/DeepFace cache under `models/`. We also pick a device: GPU if available, else CPU.

In [ ]:
# Configure model cache dirs and choose ctx_id
MODELS_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['INSIGHTFACE_HOME'] = MODELS_DIR
os.environ['DEEPFACE_HOME'] = MODELS_DIR
os.environ['ONNX_HOME'] = MODELS_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(MODELS_DIR, 'huggingface')

# Device selection similar to FaceDetector._select_ctx_id
ctx_id = -1
try:
    import torch
    torch_gpu = torch.cuda.is_available()
except Exception:
    torch_gpu = False

try:
    import onnxruntime as ort
    ort_gpu = ('CUDAExecutionProvider' in ort.get_available_providers()) or ('TensorrtExecutionProvider' in ort.get_available_providers())
except Exception:
    ort_gpu = False

if torch_gpu or ort_gpu:
    ctx_id = 0
print(f"Chosen InsightFace ctx_id: {ctx_id} ({'GPU' if ctx_id==0 else 'CPU'})")

## 4. Initialize and Inspect the SQLite Database
We will point to `database_db/face_profiles.db` by default and ensure tables are present.

In [ ]:
# Initialize DB and preview tables
DB_PATH = os.path.join(ROOT, 'database_db', 'face_profiles.db')
print("DB Path:", DB_PATH)

db = DatabaseManager(db_path=DB_PATH)

import sqlite3
conn = sqlite3.connect(DB_PATH)
print("Tables:")
for (name,) in conn.execute("SELECT name FROM sqlite_master WHERE type='table'"):
    print(" -", name)

import pandas as pd
try:
    df_profiles = pd.read_sql_query("SELECT * FROM profiles LIMIT 5", conn)
    df_enc = pd.read_sql_query("SELECT * FROM face_encodings LIMIT 5", conn)
    df_det = pd.read_sql_query("SELECT * FROM detections LIMIT 5", conn)
    display(df_profiles)
    display(df_enc)
    display(df_det)
finally:
    conn.close()

## 5. Notebook Image Display Helper (BGR → RGB)
OpenCV returns BGR images; this helper displays them correctly in Jupyter.

In [ ]:
# Matplotlib helper for BGR images
import matplotlib.pyplot as plt

def show_bgr(img, title=None, size=(6,4)):
    if img is None:
        print("No image to display")
        return
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size)
    if title:
        plt.title(title)
    plt.imshow(rgb)
    plt.axis('off')

print("Helper show_bgr() ready")

## 6. Load FaceDetector and Run on a Synthetic Test Image
We use `create_test_image()` to simulate detections and exercise the pipeline.

In [ ]:
# Build detector and run on a synthetic image
fd = FaceDetector()
img = create_test_image()
faces = fd.process_frame(img)
print(f"Detected {len(faces)} face(s)")
for i, face in enumerate(faces):
    print(f"Face {i+1}: keys={list(face.keys())}")
show_bgr(img, title="Synthetic Test Image")

## 7. Visualize Detections and Extract a Faces Grid
Draw bounding boxes, landmarks, and angle info, and show a grid of cropped faces.

In [ ]:
viz = fd.visualize_detection(img, faces)
show_bgr(viz, title="Detections with Landmarks/Angle")
faces_grid = fd.extract_faces_grid(img, faces)
show_bgr(faces_grid, title="Faces Grid", size=(6,6))

## 8. Demographics with DeepFace (Optional)
Run attribute analysis on one cropped face; this can be slower and may use CPU.

In [ ]:
attrs = None
try:
    if faces:
        fa = FaceAnalyzer()
        attrs = fa.analyze_face(faces[0]['face_img'])
        print("Attributes:", attrs)
    else:
        print("No faces to analyze")
except Exception as e:
    print("DeepFace analysis skipped:", e)

## 9. End-to-End: Detect, Analyze, and Persist to Database
We'll take one face, create a profile, store its encoding and a snapshot, and record a detection.

In [ ]:
pid = None
if faces:
    f0 = faces[0]
    yaw = float(f0.get('pose', [0])[0]) if f0.get('pose') is not None else 0.0
    # Create profile with optional demographics
    g = (attrs or {}).get('gender', 'Unknown')
    a = (attrs or {}).get('age', 'Unknown')
    e = (attrs or {}).get('ethnicity', 'Unknown')
    pid = db.add_profile(name="Workshop_User", gender=g, age_range=str(a), ethnicity=e, notes="Created in notebook")
    db.add_face_encoding(pid, f0['embedding'], angle=int(yaw))
    # Save snapshot
    added_dir = os.path.join(os.path.dirname(DB_PATH), 'added_faces')
    os.makedirs(added_dir, exist_ok=True)
    x1,y1,x2,y2 = map(int, f0['bbox'])
    crop = img[y1:y2, x1:x2]
    snap = os.path.join(added_dir, f"face_{pid}.jpg")
    if crop.size>0:
        cv2.imwrite(snap, crop)
    db.record_detection(pid, confidence=float(f0.get('det_score',0.0)), angle=int(yaw), image_path=snap)
    print(f"Profile ID {pid} created with snapshot {snap}")
else:
    print("No faces to persist")

## 10. Similarity and Matching Demo with Cosine Similarity
We’ll compute cosine similarity between a query and known encodings from the DB.

The cosine similarity formula:

$s = \frac{x \cdot y}{\|x\|\,\|y\|}$

In [ ]:
import numpy as np

def batch_cosine_similarity(embedding, embeddings):
    embedding = embedding.astype(np.float32)
    embeddings = np.array(embeddings, dtype=np.float32)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)
    dot = np.dot(embeddings, embedding)
    norm_emb = np.linalg.norm(embeddings, axis=1)
    norm_query = np.linalg.norm(embedding)
    return dot / (norm_emb * norm_query + 1e-8)

if pid is not None:
    # Load encodings for this profile (and pretend we have a gallery)
    encs = db.get_profile_encodings(pid)
    gallery = [e for e, ang in encs]
    if gallery:
        q = gallery[0]
        scores = batch_cosine_similarity(q, gallery)
        print("Scores vs. self gallery:", scores)
        print("Max score:", float(scores.max()))
    else:
        print("No encodings in DB yet for this profile")
else:
    print("No profile created; skipping match demo")

## 11. Batch Process a Folder of Images
Demo: iterate a folder, detect, analyze, and persist. Errors per-file are caught so the batch continues.

In [ ]:
import glob, time

def process_image_file(path):
    try:
        im = cv2.imread(path)
        if im is None:
            print("Skip (unreadable):", path)
            return 0
        fs = fd.process_frame(im)
        added = 0
        for f in fs:
            # Simple auto-enroll as Unknown
            yaw = float(f.get('pose', [0])[0]) if f.get('pose') is not None else 0.0
            pid_local = db.add_profile(name=f"Unknown_{int(time.time())}")
            db.add_face_encoding(pid_local, f['embedding'], int(yaw))
            # Snapshot
            added_dir = os.path.join(os.path.dirname(DB_PATH), 'added_faces')
            os.makedirs(added_dir, exist_ok=True)
            x1,y1,x2,y2 = map(int, f['bbox'])
            crop = im[y1:y2, x1:x2]
            snap = os.path.join(added_dir, f"face_{pid_local}.jpg")
            if crop.size>0:
                cv2.imwrite(snap, crop)
            db.record_detection(pid_local, float(f.get('det_score',0.0)), int(yaw), snap)
            added += 1
        return added
    except Exception as e:
        print("Error processing", path, e)
        return 0

# Example: point to a small folder of images (customize or leave empty to skip)
BATCH_DIR = os.path.join(ROOT, 'data', 'known_faces')
paths = sorted(glob.glob(os.path.join(BATCH_DIR, '**', '*.*'), recursive=True))[:10]
print("Batch candidates:", len(paths))
count = 0
for p in paths:
    count += process_image_file(p)
print("Batch added profiles:", count)

## 12. Video Frame Sampling with Auto-Enrollment
Sample every Nth frame from a video, detect, match via cosine similarity, and auto-enroll unknowns.

In [ ]:
VIDEO_PATH = os.path.join(ROOT, 'data', 'Test', 'sample.mp4')
N = 30  # sample every 30th frame

# Cache known encodings for matching
profiles = db.get_all_profiles()
known_encs = []
known_ids = []
for p in profiles:
    for emb, ang in db.get_profile_encodings(p['id']):
        known_encs.append(emb.astype(np.float32))
        known_ids.append(p['id'])

added_count = 0
if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    idx = 0
    while True:
        ret, frm = cap.read()
        if not ret:
            break
        idx += 1
        if idx % N != 0:
            continue
        fs = fd.process_frame(frm)
        for f in fs:
            emb = f['embedding']
            match_found = False
            pid_match = None
            if len(known_encs):
                scores = batch_cosine_similarity(emb, np.array(known_encs))
                if scores.size:
                    b = float(scores.max()); j = int(np.argmax(scores))
                    if b > 0.2:
                        match_found = True
                        pid_match = known_ids[j]
            if not match_found:
                yaw = float(f.get('pose', [0])[0]) if f.get('pose') is not None else 0.0
                pid_new = db.add_profile(name=f"AutoEnroll_{idx}")
                db.add_face_encoding(pid_new, emb, int(yaw))
                x1,y1,x2,y2 = map(int, f['bbox'])
                crop = frm[y1:y2, x1:x2]
                added_dir = os.path.join(os.path.dirname(DB_PATH), 'added_faces')
                os.makedirs(added_dir, exist_ok=True)
                snap = os.path.join(added_dir, f"face_{pid_new}.jpg")
                if crop.size>0:
                    cv2.imwrite(snap, crop)
                db.record_detection(pid_new, float(f.get('det_score',0.0)), int(yaw), snap)
                known_encs.append(emb.astype(np.float32))
                known_ids.append(pid_new)
                added_count += 1
    cap.release()
    print("Video auto-enroll added:", added_count)
else:
    print("Video file not found; skipping.")

## 13. Query and Visualize Database Contents
Create pandas DataFrames and visualize a simple presence timeline and demographics counts.

In [ ]:
import plotly.express as px
import pandas as pd

conn = sqlite3.connect(DB_PATH)
try:
    df_prof = pd.read_sql_query("SELECT * FROM profiles", conn)
    df_det = pd.read_sql_query("SELECT * FROM detections", conn)
finally:
    conn.close()

print("Profiles:", len(df_prof), "Detections:", len(df_det))

if not df_det.empty and 'timestamp' in df_det.columns:
    df_det['timestamp'] = pd.to_datetime(df_det['timestamp'], errors='coerce')
    dff = df_det.dropna(subset=['timestamp'])
    fig = px.scatter(dff, x='timestamp', y='profile_id', color='profile_id', title='Detections Over Time')
    fig.show()

if not df_prof.empty:
    for col in ['gender','ethnicity','age_range']:
        vc = df_prof[col].fillna('Unknown').value_counts()
        print(f"\n{col} counts:\n", vc)

## 14. Pose and Angle Handling
Extract yaw from `face.pose` and store as angle. Plot a histogram of angles stored in DB.

In [ ]:
conn = sqlite3.connect(DB_PATH)
angles = []
try:
    cur = conn.execute("SELECT angle FROM face_encodings")
    angles = [row[0] for row in cur.fetchall() if row[0] is not None]
finally:
    conn.close()

if len(angles):
    plt.figure(figsize=(6,4))
    plt.hist(angles, bins=21, color='teal', alpha=0.7)
    plt.title('Histogram of Stored Angles (Yaw)')
    plt.xlabel('Angle (deg)')
    plt.ylabel('Count')
    plt.show()
else:
    print("No angles found in DB yet.")

## 15. Sanity Checks (Mini Unit Tests)
Quick asserts to validate core invariants so the notebook stays healthy.

In [ ]:
# Assertions for core expectations
if faces:
    emb0 = faces[0]['embedding']
    assert isinstance(emb0, np.ndarray), "Embedding should be numpy array"
    assert emb0.ndim == 1, "Embedding should be 1-D vector"

# Cosine scores in [-1, 1]
if faces and len(faces) > 1:
    s = batch_cosine_similarity(faces[0]['embedding'], np.array([faces[1]['embedding']]))
    assert np.all(s <= 1.0 + 1e-5) and np.all(s >= -1.0 - 1e-5)

# DB round-trip encodings
if pid is not None:
    encs = db.get_profile_encodings(pid)
    assert len(encs) >= 1

# Snapshot exists
if pid is not None:
    snap = os.path.join(os.path.dirname(DB_PATH), 'added_faces', f'face_{pid}.jpg')
    assert os.path.exists(snap), "Snapshot image not found"

print("Sanity checks passed.")

## 16. Prepare Artifacts for the Streamlit WebApp
Confirm that the DB path and snapshots are ready for the app and show how to start it.

In [ ]:
print("DB ready at:", DB_PATH)
added_dir = os.path.join(os.path.dirname(DB_PATH), 'added_faces')
print("Snapshots dir:", added_dir, "exists=", os.path.isdir(added_dir))

print("\nRun the webapp from the project root in a terminal:")
print("  streamlit run src/sentinel.py")
print("Then navigate to Dashboard / Manage Profiles to verify newly added profiles.")

In [ ]:
# Cosine similarity helper and demo using DB encodings
import numpy as np

def batch_cosine_similarity(embedding, embeddings):
    embedding = embedding.astype(np.float32)
    embeddings = np.array(embeddings, dtype=np.float32)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)
    dot = np.dot(embeddings, embedding)
    norm_emb = np.linalg.norm(embeddings, axis=1)
    norm_query = np.linalg.norm(embedding)
    return dot / (norm_emb * norm_query + 1e-8)

# Build gallery from DB
gallery_embeddings = []
gallery_ids = []
for prof in db.get_all_profiles():
    encs = db.get_profile_encodings(prof['id'])
    for emb, ang in encs:
        gallery_embeddings.append(emb.astype(np.float32))
        gallery_ids.append(prof['id'])

if faces and gallery_embeddings:
    query = faces[0]['embedding']
    scores = batch_cosine_similarity(query, np.array(gallery_embeddings))
    best = float(scores.max()); idx = int(np.argmax(scores))
    print(f"Top score: {best:.3f} for profile_id={gallery_ids[idx]} -> {'MATCH' if best > 0.2 else 'Unknown'}")
else:
    print("Insufficient data to run matching demo.")